# exp_5 — touch-only SetSeq, fresh-seed replication (pre-registered in `PREREG.md`)

**Question (H1).** exp_4's report-only ablation `exp4_touch_ens3` (SetSeq + exp_3's 4 touch features, no aux head)
beat the exp_2 3-seed ensemble `exp3_ref_ens3` on provider labels (`target_like` +0.015 [+0.007, +0.027]). Is that
seed luck? exp_5 re-runs the identical recipe with **fresh seeds 20261111 / 12 / 13** on the same 7 IDSSE matches.

**H1 passes iff both hold** (provider labels, `target_like`, nested re-selection): the paired 95 % CI lower bound of
Δ = `exp5_touch_ens3` − `exp3_ref_ens3` is > 0, **and** mean Δ exceeds the `exp5_touch` single-seed spread on that cell.

**Scope.** Same 7 matches that selected this arm: a pass rules out seed luck, not selection on these matches.


In [1]:
from pathlib import Path
import os, sys, json, time
from datetime import datetime, timezone
ROOT = Path.cwd()
if not (ROOT / 'exp5_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_5')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import exp5_common as C
import numpy as np
import pandas as pd
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
import exp1_common as C1
if not C.PROTOCOL.exists():
    C.freeze_prereg()
    print('PREREG frozen now (before any exp_5 fit)')
protocol = C.verify_prereg()
print('PREREG verified', protocol['prereg_sha256'][:16], '| frozen', protocol['frozen_at_utc'])
guard_path = C.RESULTS / 'write_guard_before.json'
if not guard_path.exists():
    C1.write_json(guard_path, {'taken_at_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
                               'folders': C.write_manifest()})
print('write guard (exp_0-exp_4, data/):', {k: v['files'] for k, v in C1.read_json(guard_path)['folders'].items()})
import exp5_protocol as P, exp5_eval as E, exp2_protocol as P2, exp3_protocol as P3, exp3_touch as S

PREREG frozen now (before any exp_5 fit)
PREREG verified dc28ebfcacfa2676 | frozen 2026-11-10T09:06:10+00:00


write guard (exp_0-exp_4, data/): {'exp_0': 42630, 'exp_1': 215, 'exp_2': 312, 'exp_3': 407, 'exp_4': 638, 'data': 12}


## 1. Unit tests (synthetic arrays)
Arm table, nothing written under exp_0–exp_4 (tiny fit + evaluation with patched exp_5 paths), `fit=False` refusal, the two-clause decision rule, the ctrl-file gate, fingerprints.

In [2]:
import subprocess
result = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'tests'],
                        capture_output=True, text=True, cwd=ROOT)
print(result.stdout.strip().splitlines()[-1])
assert result.returncode == 0, 'unit tests failed'

16 passed, 1 warning in 8.82s


## 2. Inputs read in place
exp_2 caches (fingerprint-checked), exp_3 touch-feature caches, provider training targets (cached tokens' state / boundary must equal `exp3_protocol.step_labels(·, 'provider')`), and exp_4's touch caches (SHA-256 of all 84 `.npz` verified by `verify_prereg`).

In [3]:
for arm, fp in E.EXP2_FINGERPRINTS.items():
    missing = [k for k in C.job_names() if not (P2.prediction_folder(arm) / f'{k}.npz').exists()]
    print(f'exp_2 {arm:12s} fingerprint {fp} | missing jobs {len(missing)}')
    assert not missing and P2.fingerprint(arm) == fp
jobs = [(n, k, r) for n in C.MATCH_IDS for r, k in P.RECIPE]
assert all(S.feature_path(n, k, r).exists() for n, k, r in jobs + [(n, 'target_like', 0) for n in C.MATCH_IDS])
same = []
for name, kind, round_index in jobs:
    tokens = P2.load_tokens(name, kind, round_index)
    state, boundary = P3.step_labels(name, 'provider', tokens)
    same.append(bool(np.array_equal(state, tokens['state']) and np.array_equal(boundary, tokens['boundary'])))
print('touch caches present for', len(jobs) + len(C.MATCH_IDS), 'arrays | provider targets equal cached targets:', all(same), f'({len(same)} arrays)')
assert all(same)
for arm in C.EXP4_TOUCH_ARMS:
    folder = C.exp4_touch_folder(arm)
    print(f'exp_4 {arm:14s} {folder.name} | npz {sum(1 for _ in folder.glob("*.npz"))} (sha256 frozen)')

exp_2 exp2_aug     fingerprint 5cea8203869e | missing jobs 0
exp_2 exp2_aug_s1  fingerprint 01776e3c0b77 | missing jobs 0
exp_2 exp2_aug_s2  fingerprint fb137265b072 | missing jobs 0


touch caches present for 28 arrays | provider targets equal cached targets: True (21 arrays)
exp_4 exp4_touch     exp4_touch_2d42a3706658 | npz 28 (sha256 frozen)
exp_4 exp4_touch_s1  exp4_touch_s1_78a2fbc5df96 | npz 28 (sha256 frozen)
exp_4 exp4_touch_s2  exp4_touch_s2_4329112f1f34 | npz 28 (sha256 frozen)


## 3. Reused results: same code path
`exp3_ref_ens3` (exp_3) and `exp4_touch_ens3` (exp_4) are read from stored nested results. One cell of each (provider, `target_like`) is recomputed here from the cached predictions and must match exactly.

In [4]:
checks = [E.recheck('exp3_ref_ens3', 'target_like', 'provider'), E.recheck('exp4_touch_ens3', 'target_like', 'provider')]
C1.write_json(C.RESULTS / 'reference_recheck.json', checks)
for check in checks:
    print(check)
assert all(c['rows_equal'] and c['final_config_equal'] for c in checks)

{'variant': 'exp3_ref_ens3', 'view': 'target_like', 'label_set': 'provider', 'rows_equal': True, 'final_config_equal': True, 'mean_spell_f1': 0.6641362312691415, 'seconds': 90.9}
{'variant': 'exp4_touch_ens3', 'view': 'target_like', 'label_set': 'provider', 'rows_equal': True, 'final_config_equal': True, 'mean_spell_f1': 0.6792810366889773, 'seconds': 91.3}


In [5]:
print('exp_5 proxy fits so far:', P.proxy_fit_count(), '| planned', C.PLANNED_FITS, '| cap', C.FIT_CAP)
print(pd.DataFrame([{'arm': a, 'seed': s.seed, 'touch': s.touch, 'aux_enabled': s.aux.enabled,
                     'config_is_default': s.config == P.N.TrainConfig(seed=s.seed), 'fingerprint': P.fingerprint(a)}
                    for a, s in P.ARMS.items()]).to_string(index=False))

exp_5 proxy fits so far: 0 | planned {'reproduction_check': 1, 'proxy': 84, 'full_data': 3, 'rebuild': 3} | cap 100
          arm     seed  touch  aux_enabled  config_is_default  fingerprint
   exp5_touch 20261111   True        False               True a10cacb15e87
exp5_touch_s1 20261112   True        False               True 790052212544
exp5_touch_s2 20261113   True        False               True 65d100808055
